### 최종 - 정상 데이터 텐서 일괄 추출 및 저장 (.npy)
학습용 데이터를 구축하기 위해 `K001` 폴더 내의 모든 `.mat` 파일을 순회하면서 STFT 변환을 수행하고, 결과를 `normal_image` 폴더에 저장하는 코드입니다.

* 딥러닝 모델에 직접 입력하기 위해 값의 손실이 없는 **`.npy` (Numpy 배열) 형태**로 저장합니다.
* 눈으로 확인할 수 있도록 **`.png` 이미지 파일**도 함께 저장합니다.
* 4초(256,000개)짜리 데이터를 **1초(64,000개)씩 4등분(Chunk)**하여 데이터 개수를 4배로 늘립니다!


In [ ]:
import glob
import os
import numpy as np
from scipy import signal
import scipy.io as sio


In [ ]:
Fs = 64000

def extract_sensor_data(obj):
    arrays = []
    if isinstance(obj, np.ndarray):
        if obj.dtype.names is not None:
            for name in obj.dtype.names:
                arrays.extend(extract_sensor_data(obj[name]))
        elif obj.dtype == object:
            for item in obj.flatten():
                arrays.extend(extract_sensor_data(item))
        else:
            if obj.size > 10000 and (obj.ndim == 1 or obj.shape[0] == 1 or obj.shape[1] == 1):
                arrays.append(obj.flatten())
    elif isinstance(obj, dict):
        for k, v in obj.items():
            if not k.startswith('__'):
                arrays.extend(extract_sensor_data(v))
    return arrays

def preprocess_stft_for_cnn(sensor_data_3ch, fs, nperseg=256, noverlap=128):
    """
    핵심 주의점 1 반영: 개별 정규화가 아닌 '글로벌 정규화' 수행
    """
    # 1. 3상 전체의 STFT 결과를 일단 모아둡니다.
    all_mag_db = []
    f_out, t_out = None, None
    
    for i in range(3):
        f, t, Zxx = signal.stft(sensor_data_3ch[:, i], fs=fs, nperseg=nperseg, noverlap=noverlap)
        mag = np.abs(Zxx)
        mag_db = 20 * np.log10(mag + 1e-10) # dB 변환
        
        all_mag_db.append(mag_db)
        if f_out is None: f_out, t_out = f, t
            
    # 2. 리스트를 3D 배열(Freq, Time, 3 Channels)로 변환
    stft_3d = np.stack(all_mag_db, axis=-1)
    
    # 3. 글로벌 정규화 (전체 3상 데이터 통틀어 가장 큰 값/작은 값 기준)
    global_max = np.max(stft_3d)
    global_min = np.min(stft_3d)
    
    stft_image_normalized = (stft_3d - global_min) / (global_max - global_min)
    
#    return f_out, t_out, stft_image_normalized

In [ ]:
# K001

# # 1. 저장할 폴더 생성
save_dir = 'normal_image'
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)

# 2. K001 폴더 안의 모든 .mat 파일 검색
mat_files = glob.glob('K001/*.mat')
print(f"총 {len(mat_files)}개의 정상 .mat 파일을 찾았습니다. 변환을 시작합니다...")

# 카운터
total_saved = 0

for mat_file in mat_files:
    filename = os.path.basename(mat_file).replace('.mat', '')
    mat_data = sio.loadmat(mat_file)
    
    # 이전에 정의한 탐색 함수 사용
    sensor_arrays = extract_sensor_data(mat_data)
    
    # 가장 긴 고주파 배열(전류) 필터링
    if len(sensor_arrays) == 0: continue
    max_length = max(len(arr) for arr in sensor_arrays)
    high_freq_arrays = [arr for arr in sensor_arrays if len(arr) == max_length]
    
    if len(high_freq_arrays) >= 2:
        phase1_full = high_freq_arrays[0]
        phase2_full = high_freq_arrays[1]
        
        # 4초 데이터를 1초(64000개)씩 쪼개기 (데이터 증강)
        chunks = len(phase1_full) // 64000
        for chunk_idx in range(chunks):
            start = chunk_idx * 64000
            end = start + 64000
            
            p1_chunk = phase1_full[start:end]
            p2_chunk = phase2_full[start:end]
            p3_chunk = -(p1_chunk + p2_chunk) # 3상 복원
            
            data_3ch = np.column_stack([p1_chunk, p2_chunk, p3_chunk])
            
            # STFT 변환 및 글로벌 정규화
            _, _, stft_img = preprocess_stft_for_cnn(data_3ch, Fs, nperseg=2048, noverlap=1024)
            
            # 파일명 지정
            save_name = f"{filename}_chunk{chunk_idx}"
            
            # 1. Numpy 배열로 저장 (CNN 학습용, 데이터 손실 0%)
            np.save(os.path.join(save_dir, save_name + '.npy'), stft_img)
            
            # 2. PNG 이미지로 저장 (눈으로 확인용)
            # plt.imsave(os.path.join(save_dir, save_name + '.png'), stft_img, origin='lower')
            
            total_saved += 1
            
    print(f"[{filename}] 처리 완료")

print(f"\n 모든 변환 완료 총 {total_saved}개의 STFT 텐서와 이미지가 '{save_dir}' 폴더에 저장 완료")


총 80개의 정상 .mat 파일을 찾았습니다. 변환을 시작합니다...
[N09_M07_F10_K001_1] 처리 완료
[N09_M07_F10_K001_10] 처리 완료
[N09_M07_F10_K001_11] 처리 완료
[N09_M07_F10_K001_12] 처리 완료
[N09_M07_F10_K001_13] 처리 완료
[N09_M07_F10_K001_14] 처리 완료
[N09_M07_F10_K001_15] 처리 완료
[N09_M07_F10_K001_16] 처리 완료
[N09_M07_F10_K001_17] 처리 완료
[N09_M07_F10_K001_18] 처리 완료
[N09_M07_F10_K001_19] 처리 완료
[N09_M07_F10_K001_2] 처리 완료
[N09_M07_F10_K001_20] 처리 완료
[N09_M07_F10_K001_3] 처리 완료
[N09_M07_F10_K001_4] 처리 완료
[N09_M07_F10_K001_5] 처리 완료
[N09_M07_F10_K001_6] 처리 완료
[N09_M07_F10_K001_7] 처리 완료
[N09_M07_F10_K001_8] 처리 완료
[N09_M07_F10_K001_9] 처리 완료
[N15_M01_F10_K001_1] 처리 완료
[N15_M01_F10_K001_10] 처리 완료
[N15_M01_F10_K001_11] 처리 완료
[N15_M01_F10_K001_12] 처리 완료
[N15_M01_F10_K001_13] 처리 완료
[N15_M01_F10_K001_14] 처리 완료
[N15_M01_F10_K001_15] 처리 완료
[N15_M01_F10_K001_16] 처리 완료
[N15_M01_F10_K001_17] 처리 완료
[N15_M01_F10_K001_18] 처리 완료
[N15_M01_F10_K001_19] 처리 완료
[N15_M01_F10_K001_2] 처리 완료
[N15_M01_F10_K001_20] 처리 완료
[N15_M01_F10_K001_3] 처리 완료
[N15_M01_F10_K001_4] 

In [6]:
# K002

# # 1. 저장할 폴더 생성
save_dir = 'normal_image'
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)

# 2. K001 폴더 안의 모든 .mat 파일 검색
mat_files = glob.glob('K002/*.mat')
print(f"총 {len(mat_files)}개의 정상 .mat 파일을 찾았습니다. 변환을 시작합니다...")

# 카운터
total_saved = 0

for mat_file in mat_files:
    filename = os.path.basename(mat_file).replace('.mat', '')
    mat_data = sio.loadmat(mat_file)
    
    # 이전에 정의한 탐색 함수 사용
    sensor_arrays = extract_sensor_data(mat_data)
    
    # 가장 긴 고주파 배열(전류) 필터링
    if len(sensor_arrays) == 0: continue
    max_length = max(len(arr) for arr in sensor_arrays)
    high_freq_arrays = [arr for arr in sensor_arrays if len(arr) == max_length]
    
    if len(high_freq_arrays) >= 2:
        phase1_full = high_freq_arrays[0]
        phase2_full = high_freq_arrays[1]
        
        # 4초 데이터를 1초(64000개)씩 쪼개기 (데이터 증강)
        chunks = len(phase1_full) // 64000
        for chunk_idx in range(chunks):
            start = chunk_idx * 64000
            end = start + 64000
            
            p1_chunk = phase1_full[start:end]
            p2_chunk = phase2_full[start:end]
            p3_chunk = -(p1_chunk + p2_chunk) # 3상 복원
            
            data_3ch = np.column_stack([p1_chunk, p2_chunk, p3_chunk])
            
            # STFT 변환 및 글로벌 정규화
            _, _, stft_img = preprocess_stft_for_cnn(data_3ch, Fs, nperseg=2048, noverlap=1024)
            
            # 파일명 지정
            save_name = f"{filename}_chunk{chunk_idx}"
            
            # 1. Numpy 배열로 저장 (CNN 학습용, 데이터 손실 0%)
            np.save(os.path.join(save_dir, save_name + '.npy'), stft_img)
            
            # 2. PNG 이미지로 저장 (눈으로 확인용)
            # plt.imsave(os.path.join(save_dir, save_name + '.png'), stft_img, origin='lower')
            
            total_saved += 1
            
    print(f"[{filename}] 처리 완료")

print(f"\n 모든 변환 완료 총 {total_saved}개의 STFT 텐서와 이미지가 '{save_dir}' 폴더에 저장 완료")


총 80개의 정상 .mat 파일을 찾았습니다. 변환을 시작합니다...
[N09_M07_F10_K002_1] 처리 완료
[N09_M07_F10_K002_10] 처리 완료
[N09_M07_F10_K002_11] 처리 완료
[N09_M07_F10_K002_12] 처리 완료
[N09_M07_F10_K002_13] 처리 완료
[N09_M07_F10_K002_14] 처리 완료
[N09_M07_F10_K002_15] 처리 완료
[N09_M07_F10_K002_16] 처리 완료
[N09_M07_F10_K002_17] 처리 완료
[N09_M07_F10_K002_18] 처리 완료
[N09_M07_F10_K002_19] 처리 완료
[N09_M07_F10_K002_2] 처리 완료
[N09_M07_F10_K002_20] 처리 완료
[N09_M07_F10_K002_3] 처리 완료
[N09_M07_F10_K002_4] 처리 완료
[N09_M07_F10_K002_5] 처리 완료
[N09_M07_F10_K002_6] 처리 완료
[N09_M07_F10_K002_7] 처리 완료
[N09_M07_F10_K002_8] 처리 완료
[N09_M07_F10_K002_9] 처리 완료
[N15_M01_F10_K002_1] 처리 완료
[N15_M01_F10_K002_10] 처리 완료
[N15_M01_F10_K002_11] 처리 완료
[N15_M01_F10_K002_12] 처리 완료
[N15_M01_F10_K002_13] 처리 완료
[N15_M01_F10_K002_14] 처리 완료
[N15_M01_F10_K002_15] 처리 완료
[N15_M01_F10_K002_16] 처리 완료
[N15_M01_F10_K002_17] 처리 완료
[N15_M01_F10_K002_18] 처리 완료
[N15_M01_F10_K002_19] 처리 완료
[N15_M01_F10_K002_2] 처리 완료
[N15_M01_F10_K002_20] 처리 완료
[N15_M01_F10_K002_3] 처리 완료
[N15_M01_F10_K002_4] 